# Employee Sample Validation

This notebook examines the first 50 synthetic employees generated for the Workforce Intelligence and Retention Decision Platform.

The goals are to:

- Confirm that the files load correctly
- Inspect rows, columns, and data types
- Validate primary-key and foreign-key rules
- Check employment-status logic
- Connect identifiers to readable reference values
- Identify limitations before scaling to 10,000 employees

In [2]:
from pathlib import Path

import pandas as pd


# Determine the main project folder.
PROJECT_ROOT = Path.cwd()

# Depending on how VS Code launches the notebook,
# the current folder might be the notebooks folder.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project folder:")
print(PROJECT_ROOT)

print("\nRaw data folder:")
print(RAW_DATA_DIR)

Project folder:
c:\Users\Yuchen Xu\Documents\workforce-intelligence-platform

Raw data folder:
c:\Users\Yuchen Xu\Documents\workforce-intelligence-platform\data\raw


In [3]:
employees = pd.read_csv(
    RAW_DATA_DIR / "employees_sample.csv",
    parse_dates=["hire_date", "termination_date"],
)

departments = pd.read_csv(
    RAW_DATA_DIR / "departments.csv"
)

locations = pd.read_csv(
    RAW_DATA_DIR / "locations.csv"
)

job_roles = pd.read_csv(
    RAW_DATA_DIR / "job_roles.csv"
)

training_programs = pd.read_csv(
    RAW_DATA_DIR / "training_programs.csv"
)

print("All files loaded successfully.")

All files loaded successfully.


## 1. Table sizes

In [4]:
table_sizes = pd.DataFrame(
    {
        "table": [
            "employees",
            "departments",
            "locations",
            "job_roles",
            "training_programs",
        ],
        "rows": [
            len(employees),
            len(departments),
            len(locations),
            len(job_roles),
            len(training_programs),
        ],
        "columns": [
            len(employees.columns),
            len(departments.columns),
            len(locations.columns),
            len(job_roles.columns),
            len(training_programs.columns),
        ],
    }
)

table_sizes

,table,rows,columns
0,employees,50,14
1,departments,8,4
2,locations,5,5
3,job_roles,20,6
4,training_programs,10,5


## 2. Initial employee inspection

In [5]:
employees.head(10)

,employee_id,first_name,last_name,hire_date,termination_date,employment_status,termination_type,department_id,location_id,job_role_id,manager_id,employment_type,birth_year,education_level
0,10001,Danielle,Johnson,2021-01-26,NaT,Active,NaN,1,1,5,NaN,Salaried,1982,Bachelor's
1,10002,Joshua,Walker,2021-04-15,NaT,Active,NaN,2,2,8,NaN,Salaried,1970,Master's
2,10003,Jill,Rhodes,2021-01-31,NaT,Active,NaN,3,1,10,NaN,Salaried,1970,Bachelor's
3,10004,Patricia,Miller,2022-09-09,NaT,Active,NaN,4,5,13,NaN,Salaried,1966,Master's
4,10005,Robert,Johnson,2022-03-06,NaT,Active,NaN,5,5,15,NaN,Salaried,1979,Bachelor's
5,10006,Jeffery,Wagner,2021-01-07,NaT,Active,NaN,6,3,17,NaN,Salaried,1975,Bachelor's
6,10007,Anthony,Gonzalez,2021-10-12,NaT,Active,NaN,7,3,19,NaN,Salaried,1974,Bachelor's
7,10008,Debra,Gardner,2021-04-15,NaT,Active,NaN,8,3,20,NaN,Hourly,1970,Associate
8,10009,Jeffrey,Lawrence,2025-07-11,2026-05-02,Terminated,Voluntary,2,3,7,10002.0,Hourly,1986,High School
9,10010,Lisa,Smith,2025-10-31,NaT,Active,NaN,2,3,8,10002.0,Salaried,1974,Master's


In [6]:
employees.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   employee_id        50 non-null     int64         
 1   first_name         50 non-null     str           
 2   last_name          50 non-null     str           
 3   hire_date          50 non-null     datetime64[us]
 4   termination_date   11 non-null     datetime64[us]
 5   employment_status  50 non-null     str           
 6   termination_type   11 non-null     str           
 7   department_id      50 non-null     int64         
 8   location_id        50 non-null     int64         
 9   job_role_id        50 non-null     int64         
 10  manager_id         42 non-null     float64       
 11  employment_type    50 non-null     str           
 12  birth_year         50 non-null     int64         
 13  education_level    50 non-null     str           
dtypes: datetime64[us](2), f

In [7]:
employees["manager_id"] = employees["manager_id"].astype("Int64")

employees[["employee_id", "manager_id"]].head(10)

,employee_id,manager_id
0,10001,<NA>
1,10002,<NA>
2,10003,<NA>
3,10004,<NA>
4,10005,<NA>
5,10006,<NA>
6,10007,<NA>
7,10008,<NA>
8,10009,10002
9,10010,10002


## 3. Missing-value review

In [8]:
missing_summary = employees.isna().sum().to_frame(
    name="missing_count"
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"]
    / len(employees)
    * 100
).round(1)

missing_summary

,missing_count,missing_percent
employee_id,0,0.0
first_name,0,0.0
last_name,0,0.0
hire_date,0,0.0
termination_date,39,78.0
employment_status,0,0.0
termination_type,39,78.0
department_id,0,0.0
location_id,0,0.0
job_role_id,0,0.0


## 4. Primary-key validation

In [9]:
primary_key_checks = pd.Series(
    {
        "employee_id exists": (
            "employee_id" in employees.columns
        ),
        "employee_id has no missing values": (
            employees["employee_id"].notna().all()
        ),
        "employee_id values are unique": (
            employees["employee_id"].is_unique
        ),
    },
    name="passed",
)

primary_key_checks

employee_id exists                   True
employee_id has no missing values    True
employee_id values are unique        True
Name: passed, dtype: bool

## 5. Foreign-key validation

In [10]:
valid_department_ids = set(
    departments["department_id"]
)

valid_location_ids = set(
    locations["location_id"]
)

valid_job_role_ids = set(
    job_roles["job_role_id"]
)

valid_employee_ids = set(
    employees["employee_id"]
)

used_manager_ids = set(
    employees["manager_id"]
    .dropna()
    .astype(int)
)

foreign_key_checks = pd.Series(
    {
        "all department IDs are valid": (
            set(employees["department_id"])
            .issubset(valid_department_ids)
        ),
        "all location IDs are valid": (
            set(employees["location_id"])
            .issubset(valid_location_ids)
        ),
        "all job-role IDs are valid": (
            set(employees["job_role_id"])
            .issubset(valid_job_role_ids)
        ),
        "all manager IDs are valid employees": (
            used_manager_ids
            .issubset(valid_employee_ids)
        ),
    },
    name="passed",
)

foreign_key_checks

all department IDs are valid           True
all location IDs are valid             True
all job-role IDs are valid             True
all manager IDs are valid employees    True
Name: passed, dtype: bool

## 6. Employment-status validation

In [11]:
active_mask = (
    employees["employment_status"] == "Active"
)

terminated_mask = (
    employees["employment_status"] == "Terminated"
)

employment_status_checks = pd.Series(
    {
        "active employees have no termination date": (
            employees.loc[
                active_mask,
                "termination_date",
            ].isna().all()
        ),
        "active employees have no termination type": (
            employees.loc[
                active_mask,
                "termination_type",
            ].isna().all()
        ),
        "terminated employees have a termination date": (
            employees.loc[
                terminated_mask,
                "termination_date",
            ].notna().all()
        ),
        "terminated employees have a termination type": (
            employees.loc[
                terminated_mask,
                "termination_type",
            ].notna().all()
        ),
        "termination dates are not before hire dates": (
            (
                employees.loc[
                    terminated_mask,
                    "termination_date",
                ]
                >=
                employees.loc[
                    terminated_mask,
                    "hire_date",
                ]
            ).all()
        ),
    },
    name="passed",
)

employment_status_checks

active employees have no termination date       True
active employees have no termination type       True
terminated employees have a termination date    True
terminated employees have a termination type    True
termination dates are not before hire dates     True
Name: passed, dtype: bool

In [12]:
self_managed = employees[
    employees["manager_id"].notna()
    & (
        employees["manager_id"]
        == employees["employee_id"]
    )
]

print(
    "Number of employees who manage themselves:",
    len(self_managed),
)

Number of employees who manage themselves: 0


## 7. Connect employee IDs to readable names

In [13]:
employee_details = (
    employees
    .merge(
        departments,
        on="department_id",
        how="left",
    )
    .merge(
        locations,
        on="location_id",
        how="left",
    )
    .merge(
        job_roles,
        on="job_role_id",
        how="left",
    )
)

selected_columns = [
    "employee_id",
    "first_name",
    "last_name",
    "department_name",
    "city",
    "state",
    "job_title",
    "employment_status",
]

employee_details[selected_columns].head(10)

,employee_id,first_name,last_name,department_name,city,state,job_title,employment_status
0,10001,Danielle,Johnson,Engineering,Austin,Texas,Senior Software Engineer,Active
1,10002,Joshua,Walker,Manufacturing,Fremont,California,Production Supervisor,Active
2,10003,Jill,Rhodes,Supply Chain,Austin,Texas,Operations Research Analyst,Active
3,10004,Patricia,Miller,Sales,Phoenix,Arizona,Account Manager,Active
4,10005,Robert,Johnson,Finance,Phoenix,Arizona,Risk Analyst,Active
5,10006,Jeffery,Wagner,Human Resources,Reno,Nevada,Recruiter,Active
6,10007,Anthony,Gonzalez,Information Technology,Reno,Nevada,Cybersecurity Risk Analyst,Active
7,10008,Debra,Gardner,Customer Support,Reno,Nevada,Customer Support Specialist,Active
8,10009,Jeffrey,Lawrence,Manufacturing,Reno,Nevada,Production Technician,Terminated
9,10010,Lisa,Smith,Manufacturing,Reno,Nevada,Production Supervisor,Active


In [14]:
join_checks = pd.Series(
    {
        "row count remained 50": (
            len(employee_details) == len(employees)
        ),
        "all employees received department names": (
            employee_details["department_name"]
            .notna()
            .all()
        ),
        "all employees received location names": (
            employee_details["city"]
            .notna()
            .all()
        ),
        "all employees received job titles": (
            employee_details["job_title"]
            .notna()
            .all()
        ),
    },
    name="passed",
)

join_checks

row count remained 50                      True
all employees received department names    True
all employees received location names      True
all employees received job titles          True
Name: passed, dtype: bool

## 8. Derived employee information

In [15]:
AS_OF_DATE = pd.Timestamp("2026-06-30")

employee_details["age"] = (
    AS_OF_DATE.year
    - employee_details["birth_year"]
)

employee_end_date = (
    employee_details["termination_date"]
    .fillna(AS_OF_DATE)
)

employee_details["tenure_days"] = (
    employee_end_date
    - employee_details["hire_date"]
).dt.days

employee_details["tenure_years"] = (
    employee_details["tenure_days"]
    / 365.25
).round(2)

employee_details[
    [
        "employee_id",
        "hire_date",
        "termination_date",
        "employment_status",
        "age",
        "tenure_years",
    ]
].head(10)

,employee_id,hire_date,termination_date,employment_status,age,tenure_years
0,10001,2021-01-26,NaT,Active,44,5.42
1,10002,2021-04-15,NaT,Active,56,5.21
2,10003,2021-01-31,NaT,Active,56,5.41
3,10004,2022-09-09,NaT,Active,60,3.81
4,10005,2022-03-06,NaT,Active,47,4.32
5,10006,2021-01-07,NaT,Active,51,5.48
6,10007,2021-10-12,NaT,Active,52,4.71
7,10008,2021-04-15,NaT,Active,56,5.21
8,10009,2025-07-11,2026-05-02,Terminated,40,0.81
9,10010,2025-10-31,NaT,Active,52,0.66


## 9. Workforce summary

In [16]:
workforce_summary = pd.Series(
    {
        "total employees": len(employee_details),
        "active employees": (
            employee_details["employment_status"]
            .eq("Active")
            .sum()
        ),
        "terminated employees": (
            employee_details["employment_status"]
            .eq("Terminated")
            .sum()
        ),
        "voluntary terminations": (
            employee_details["termination_type"]
            .eq("Voluntary")
            .sum()
        ),
        "involuntary terminations": (
            employee_details["termination_type"]
            .eq("Involuntary")
            .sum()
        ),
        "average age": round(
            employee_details["age"].mean(),
            1,
        ),
        "average tenure in years": round(
            employee_details["tenure_years"]
            .mean(),
            2,
        ),
    },
    name="value",
)

workforce_summary

total employees             50.00
active employees            39.00
terminated employees        11.00
voluntary terminations       8.00
involuntary terminations     3.00
average age                 43.80
average tenure in years      2.61
Name: value, dtype: float64

In [17]:
department_summary = (
    employee_details
    .groupby("department_name")
    .agg(
        employee_count=(
            "employee_id",
            "count",
        ),
        active_count=(
            "employment_status",
            lambda values: (
                values == "Active"
            ).sum(),
        ),
        terminated_count=(
            "employment_status",
            lambda values: (
                values == "Terminated"
            ).sum(),
        ),
        average_tenure_years=(
            "tenure_years",
            "mean",
        ),
    )
    .reset_index()
)

department_summary[
    "terminated_share"
] = (
    department_summary["terminated_count"]
    / department_summary["employee_count"]
).round(3)

department_summary = (
    department_summary
    .sort_values(
        "employee_count",
        ascending=False,
    )
)

department_summary

,department_name,employee_count,active_count,terminated_count,average_tenure_years,terminated_share
5,Manufacturing,14,8,6,2.182857,0.429
1,Engineering,10,9,1,2.612000,0.100
4,Information Technology,8,6,2,2.266250,0.250
6,Sales,6,6,0,3.141667,0.000
7,Supply Chain,4,2,2,2.897500,0.500
0,Customer Support,3,3,0,2.860000,0.000
3,Human Resources,3,3,0,3.280000,0.000
2,Finance,2,2,0,3.350000,0.000


## 10. Manager structure

In [18]:
direct_report_counts = (
    employees["manager_id"]
    .dropna()
    .astype(int)
    .value_counts()
    .rename_axis("manager_id")
    .reset_index(name="direct_reports")
)

manager_details = (
    direct_report_counts
    .merge(
        employees[
            [
                "employee_id",
                "first_name",
                "last_name",
                "department_id",
            ]
        ],
        left_on="manager_id",
        right_on="employee_id",
        how="left",
    )
    .merge(
        departments[
            [
                "department_id",
                "department_name",
            ]
        ],
        on="department_id",
        how="left",
    )
)

manager_details[
    [
        "manager_id",
        "first_name",
        "last_name",
        "department_name",
        "direct_reports",
    ]
].sort_values(
    "direct_reports",
    ascending=False,
)

,manager_id,first_name,last_name,department_name,direct_reports
0,10002,Joshua,Walker,Manufacturing,13
1,10001,Danielle,Johnson,Engineering,9
2,10007,Anthony,Gonzalez,Information Technology,7
3,10004,Patricia,Miller,Sales,5
4,10003,Jill,Rhodes,Supply Chain,3
5,10006,Jeffery,Wagner,Human Resources,2
6,10008,Debra,Gardner,Customer Support,2
7,10005,Robert,Johnson,Finance,1


## 11. Complete validation summary

In [19]:
all_checks = pd.concat(
    [
        primary_key_checks,
        foreign_key_checks,
        employment_status_checks,
        join_checks,
    ]
)

validation_summary = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_summary

,check,passed
0,employee_id exists,True
1,employee_id has no missing values,True
2,employee_id values are unique,True
3,all department IDs are valid,True
4,all location IDs are valid,True
5,all job-role IDs are valid,True
6,all manager IDs are valid employees,True
7,active employees have no termination date,True
8,active employees have no termination type,True
9,terminated employees have a termination date,True


In [20]:
if validation_summary["passed"].all():
    print("All employee-sample validation checks passed.")
else:
    print("One or more validation checks failed.")

All employee-sample validation checks passed.


## 12. Initial conclusions

The 50-row employee sample successfully passed the initial validation checks:

- Employee IDs are complete and unique.
- Department, location, job-role, and manager IDs are valid.
- Active employees do not have termination information.
- Terminated employees have valid termination dates and types.
- No employee manages themselves.
- Reference-table joins preserve all employee rows.
- Department, location, and job-role descriptions are complete.

### Limitations identified

The current employee generator is intentionally simplified.

1. Each department currently has only one manager.
2. Fifty employees are too few for reliable statistical conclusions.
3. Employee age is calculated approximately from birth year.
4. Department assignment and termination behavior use simple predefined probabilities.
5. There is no historical record of promotions, transfers, manager changes, salary changes, or training yet.
6. The current terminated share is not a formal annual turnover rate.

The next version should improve the management hierarchy before scaling the employee population.